## Updated Evaluation Cells — Section 7

Replace your existing Section 7 evaluation cells with these.
Three changes: fixed random seed, 1000 users, narrowed Optuna search.


In [ ]:
def prec_at_k(rec,rel,k): return len(set(rec[:k])&rel)/k if k else 0
def rec_at_k(rec,rel,k):  return len(set(rec[:k])&rel)/len(rel) if rel else 0
def f1(p,r): return 2*p*r/(p+r) if p+r else 0

K_VALUES = [1,3,5,10]

# Fix random seed BEFORE building eval sets — makes results reproducible
random.seed(42)
np.random.seed(42)

train_by_user = {str(k):{str(r) for r in v}
                  for k,v in df_train.groupby('user_id')['recipe_id'].apply(set).items()}
test_by_user  = {str(k):{str(r) for r in v}
                  for k,v in df_test.groupby('user_id')['recipe_id'].apply(set).items()}
known_rids = set(R2I.keys())

eval_users=[]; eval_test={}
for u in test_by_user:
    if u not in train_by_user or u not in U2I: continue
    kp={r for r in test_by_user[u] if r in known_rids}
    if kp: eval_users.append(u); eval_test[u]=kp

active_users = [u for u in eval_users if len(train_by_user.get(u,set()))>=10]
print(f'Eval users: {len(eval_users):,}  |  Active: {len(active_users):,}')
validate(len(active_users)>100,'Sufficient active users','>100',f'{len(active_users):,}')


In [ ]:
def eval_method1(score_fn, label, max_users=1000):  # increased from 500 → 1000
    random.seed(42); np.random.seed(42)  # fix seed inside function too
    prec={k:[] for k in K_VALUES}; rec={k:[] for k in K_VALUES}
    for uid in active_users[:max_users]:
        seen=train_by_user.get(uid,set()); all_pos=eval_test[uid]
        for pos in list(all_pos)[:3]:
            neg_pool=list(known_rids-seen-all_pos)
            if len(neg_pool)<99: continue
            candidates=[pos]+random.sample(neg_pool,99)
            preds=sorted([(r,score_fn(uid,r)) for r in candidates],
                          key=lambda x:x[1],reverse=True)
            rec_ids=[p[0] for p in preds]; rel={pos}
            for k in K_VALUES:
                prec[k].append(prec_at_k(rec_ids,rel,k))
                rec[k].append(rec_at_k(rec_ids,rel,k))
    print(f'{label}  (n={len(prec[1])} samples):')
    print(f'{"k":<5}{"Precision":>12}{"Recall":>12}')
    for k in K_VALUES:
        print(f'{k:<5}{np.mean(prec[k]):>12.4f}{np.mean(rec[k]):>12.4f}')
    return prec,rec

def eval_method2(score_fn, label, max_users=1000):  # increased from 500 → 1000
    random.seed(42); np.random.seed(42)
    prec={k:[] for k in K_VALUES}; rec={k:[] for k in K_VALUES}
    for uid in active_users[:max_users]:
        seen=train_by_user.get(uid,set())
        pos_list=list(eval_test.get(uid,set()))
        if not pos_list: continue
        held=pos_list[0]
        neg_pool=list(known_rids-seen-{held})
        if len(neg_pool)<99: continue
        candidates=[held]+random.sample(neg_pool,99)
        preds=sorted([(r,score_fn(uid,r)) for r in candidates],
                      key=lambda x:x[1],reverse=True)
        rec_ids=[p[0] for p in preds]; rel={held}
        for k in K_VALUES:
            prec[k].append(prec_at_k(rec_ids,rel,k))
            rec[k].append(rec_at_k(rec_ids,rel,k))
    print(f'{label}  (n={len(prec[1])} samples):')
    for k in K_VALUES:
        print(f'  k={k:<3} P={np.mean(prec[k]):.4f}  R={np.mean(rec[k]):.4f}')
    return prec,rec


In [ ]:
print('=== SVD — Method 1 ===')
svd_prec,svd_rec = eval_method1(
    lambda uid,rid: svd.predict(U2I[uid],R2I[rid]).est,'SVD')

print('\n=== BPR — Method 1 ===')
bpr_prec,bpr_rec = eval_method1(bpr_score,'BPR (default)')

validate(np.mean(svd_prec[1])>0,'SVD Precision@1 non-zero','>0',f'{np.mean(svd_prec[1]):.4f}')
validate(np.mean(bpr_prec[1])>0,'BPR Precision@1 non-zero','>0',f'{np.mean(bpr_prec[1]):.4f}')

print('\n=== SVD — Method 2 (LOO) ===')
svd_prec2,svd_rec2 = eval_method2(
    lambda uid,rid: svd.predict(U2I[uid],R2I[rid]).est,'SVD')

print('\n=== BPR — Method 2 (LOO) ===')
bpr_prec2,bpr_rec2 = eval_method2(bpr_score,'BPR (default)')


In [ ]:
print('=== Algorithm Selection ===')
print(f'{"k":<5}{"SVD":>10}{"BPR":>10}{"Winner":>10}')
for k in K_VALUES:
    s,b=np.mean(svd_prec[k]),np.mean(bpr_prec[k])
    print(f'{k:<5}{s:>10.4f}{b:>10.4f}{"BPR" if b>s else "SVD":>10}')

ACTIVE_CF_MODEL='bpr' if np.mean(bpr_prec[1])>np.mean(svd_prec[1]) else 'svd'
print(f'\n🏆 Selected for tuning: {ACTIVE_CF_MODEL.upper()}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(K_VALUES,[np.mean(svd_prec[k]) for k in K_VALUES],
         marker='o',color=C_PURPLE,linewidth=2,markersize=7,label='SVD')
ax.plot(K_VALUES,[np.mean(bpr_prec[k]) for k in K_VALUES],
         marker='^',color=C_AFTER,linewidth=2,markersize=7,label='BPR')
ax.plot(K_VALUES,[k/100 for k in K_VALUES],
         marker='s',color=C_FLAG,linestyle='--',markersize=5,label='Random')
ax.set_xlabel('k'); ax.set_ylabel('Precision@k')
ax.set_title('Method 1: Precision@k',fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(K_VALUES,[np.mean(svd_rec2[k]) for k in K_VALUES],
         marker='o',color=C_PURPLE,linewidth=2,markersize=7,label='SVD')
ax.plot(K_VALUES,[np.mean(bpr_rec2[k]) for k in K_VALUES],
         marker='^',color=C_AFTER,linewidth=2,markersize=7,label='BPR')
ax.axhline(np.mean(svd_rec2[1]),color=C_FLAG,linestyle='--',linewidth=1.5,
            label=f'SVD Recall (flat at {np.mean(svd_rec2[1]):.0%})')
ax.set_xlabel('k'); ax.set_ylabel('Recall@k')
ax.set_title('Method 2: Recall@k (LOO)',fontweight='bold')
ax.legend(fontsize=8)

plt.suptitle('SVD vs BPR — Precision@k and Recall@k',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('plots/svd_vs_bpr.png',dpi=120,bbox_inches='tight')
plt.show()


### Section 8 — Updated Optuna Cell
Replace your existing Optuna objective with this narrowed search.


In [ ]:
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_USERS = active_users[:100]

# Narrowed search space around known good region from previous run
# Previous best: n_factors=75, reg=0.003, lr=0.024
if ACTIVE_CF_MODEL=='svd':
    def objective(trial):
        nf  = trial.suggest_int('n_factors', 50, 100)           # narrowed around 75
        reg = trial.suggest_float('reg_all',  0.001, 0.02, log=True)  # narrowed around 0.003
        lr  = trial.suggest_float('lr_all',   0.01,  0.05, log=True)  # narrowed around 0.024
        m   = SVD(n_factors=nf,reg_all=reg,lr_all=lr,n_epochs=20,random_state=42)
        m.fit(trainset)
        random.seed(42)  # fix seed for stable tuning
        hits=[]
        for uid in TUNE_USERS:
            if uid not in U2I: continue
            seen=train_by_user.get(uid,set())
            pos_list=list(eval_test.get(uid,set()))
            if not pos_list: continue
            pos=pos_list[0]
            neg_pool=list(known_rids-seen-eval_test.get(uid,set()))
            if len(neg_pool)<99: continue
            cands=[pos]+random.sample(neg_pool,99)
            preds=sorted([(r,m.predict(U2I[uid],R2I[r]).est) for r in cands],
                          key=lambda x:x[1],reverse=True)
            hits.append(int([p[0] for p in preds].index(pos)+1<=10))
        return np.mean(hits) if hits else 0.0
else:
    def objective(trial):
        kf    = trial.suggest_int('k',16,128)
        lr    = trial.suggest_float('learning_rate',0.001,0.1,log=True)
        reg   = trial.suggest_float('lambda_reg',0.0001,0.05,log=True)
        iters = trial.suggest_int('max_iter',50,200)
        m = BPR(k=kf,max_iter=iters,learning_rate=lr,lambda_reg=reg,seed=42,verbose=False)
        m.fit(cornac_data)
        random.seed(42)
        hits=[]
        for uid in TUNE_USERS:
            seen=train_by_user.get(uid,set())
            pos_list=list(eval_test.get(uid,set()))
            if not pos_list: continue
            pos=pos_list[0]
            neg_pool=list(known_rids-seen-eval_test.get(uid,set()))
            if len(neg_pool)<99: continue
            cands=[pos]+random.sample(neg_pool,99)
            try:
                preds=sorted([(r,m.score(cornac_data.uid_map[uid],cornac_data.iid_map[r]))
                              for r in cands if r in cornac_data.iid_map],
                              key=lambda x:x[1],reverse=True)
                hits.append(int([p[0] for p in preds].index(pos)+1<=10))
            except: continue
        return np.mean(hits) if hits else 0.0

print(f'Running Optuna for {ACTIVE_CF_MODEL.upper()} (30 trials, narrowed search)...')
study=optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective,n_trials=30)

best=study.best_params
print(f'Best Hit@10: {study.best_value:.1%}')
for k,v in best.items(): print(f'  {k:<15}={v}')
